# 05 · Package Images

Packs all referenced images into a single array file on Drive. Because the images
are uniform 224×224 grayscale, the entire dataset is stored as one contiguous
`uint8` array. Subsequent sessions transfer this single file (~3 GB) rather than
tens of thousands of individual files, and training reads pixels directly from
memory without per-image decoding. Images are read from Drive in parallel to hide
per-file network latency. Executed a single time.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import pandas as pd, numpy as np, os, time, shutil, threading
import concurrent.futures as cf
from pathlib import Path
from PIL import Image
manifest=pd.read_csv(str(config.MANIFEST_CSV))
N=len(manifest); S=config.IMG_SIZE
print(f'Manifest: {N:,} images -> array ({N}, {S}, {S}) uint8 = {N*S*S/1e9:.2f} GB')

Mounted at /content/drive
Manifest: 61,591 images -> array (61591, 224, 224) uint8 = 3.09 GB


## Parallel read into a preallocated array

A single array of shape (N, 224, 224) is preallocated. Worker threads read each
image from Drive and write it into its assigned row, so memory is not duplicated.
Concurrency hides the per-file network latency that dominates sequential reads.

In [2]:
arr=np.empty((N, S, S), dtype=np.uint8)
rows=manifest.reset_index(drop=True)
paths=[str(config.DATASETS[r['dataset']]['png_dir']/r['filename']) for _,r in rows.iterrows()]

print(f'Reading {N:,} images (16 workers)...')
t0=time.time(); done=[0]; failed=[]; lock=threading.Lock()
def load(i):
    try:
        im=Image.open(paths[i]).convert('L')
        if im.size!=(S,S): im=im.resize((S,S), Image.LANCZOS)
        arr[i]=np.array(im, dtype=np.uint8)
        with lock:
            done[0]+=1
            if done[0]%5000==0:
                el=time.time()-t0; print(f'  {done[0]:,}/{N:,} ({done[0]/el:.0f}/s, {el:.0f}s)',flush=True)
        return True
    except Exception:
        with lock: failed.append(i)
        return False
with cf.ThreadPoolExecutor(max_workers=16) as ex:
    list(ex.map(load, range(N)))
print(f'Read {N-len(failed):,}/{N:,} in {time.time()-t0:.0f}s; failed {len(failed)}')

if failed:
    keep=np.array([i for i in range(N) if i not in set(failed)])
    arr=arr[keep]; rows=rows.iloc[keep].reset_index(drop=True); N=len(rows)
    print(f'After dropping failures: {N:,} images')

Reading 61,591 images (16 workers)...
  5,000/61,591 (18/s, 270s)
  10,000/61,591 (19/s, 526s)
  15,000/61,591 (19/s, 780s)
  20,000/61,591 (21/s, 965s)
  25,000/61,591 (17/s, 1466s)
  30,000/61,591 (18/s, 1707s)
  35,000/61,591 (18/s, 1948s)
  40,000/61,591 (18/s, 2188s)
  45,000/61,591 (19/s, 2428s)
  50,000/61,591 (19/s, 2668s)
  55,000/61,591 (19/s, 2909s)
  60,000/61,591 (19/s, 3148s)
Read 61,558/61,591 in 3222s; failed 33
After dropping failures: 61,558 images


## Save array and index

The array is written locally then copied to Drive as one file. An `arr_idx` column
recording each image's row position is added to the manifest, allowing any data
subset (training, validation, test) to index the array directly.

In [3]:
rows['arr_idx']=np.arange(len(rows))
rows.to_csv(str(config.MANIFEST_CSV), index=False)
print(f'Manifest updated with arr_idx: {len(rows):,} rows')

tmp='/content/images.npy'
print('Saving array locally...')
t1=time.time(); np.save(tmp, arr); print(f'Saved in {time.time()-t1:.0f}s ({os.path.getsize(tmp)/1e9:.2f} GB)')
print('Copying to Drive...')
t2=time.time(); shutil.copy(tmp, str(config.IMAGES_NPY)); print(f'Uploaded in {time.time()-t2:.0f}s')
print(f'Array on Drive: {config.IMAGES_NPY}')
print('Run once. Training notebooks load this single file per session.')

Manifest updated with arr_idx: 61,558 rows
Saving array locally...
Saved in 116s (3.09 GB)
Copying to Drive...
Uploaded in 30s
Array on Drive: /content/drive/MyDrive/Master Thesis/scope3/images.npy
Run once. Training notebooks load this single file per session.
